# Paper 31: Conformer — Convolution-augmented Transformer for Speech Recognition
## A clear NumPy forward pass with a CTC output head

Conformer combines two useful views of speech:

- **Self-attention** connects information across the whole utterance.
- **Convolution** recognizes local patterns such as short sound transitions.

This notebook builds a small Conformer-CTC model from basic NumPy operations. The weights are random, so the goal is to understand the architecture and tensor shapes—not to recognize real speech.

Adapted from the repository's `demo_conformerCTC.py`.


In [ ]:
import numpy as np

np.random.seed(0)

# Shape notation used throughout the notebook:
# T = number of time frames
# F = input feature dimension
# D = model dimension
# H = number of attention heads


## Goal and Architecture

We will follow one utterance through this pipeline:

```text
acoustic features [T, F]
        ↓
two stride-2 Conv2D layers
        ↓
subsampled sequence [T', D]
        ↓
Conformer blocks [T', D]
        ↓
linear CTC head
        ↓
frame logits [T', vocabulary_size]
        ↓
greedy CTC collapse
        ↓
token IDs
```

The two convolution layers reduce time by about four times. The encoder then keeps the shorter time dimension unchanged.


## Small Numerical Building Blocks

The model needs stable softmax, Swish activation, normalization, and linear layers. Layer normalization uses statistics across the feature dimension of each frame. The simplified batch normalization later in the notebook uses statistics across time for each channel.


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def swish(x):
    return x * sigmoid(x)


def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)
#numerically softmax([1, 2, 3]) = softmax([-2, -1, 0]), do this is to fix the number overflow, e^x increase very fast.

def layer_norm(x, gamma, beta):
    mean = x.mean(axis=-1, keepdims=True)
    variance = x.var(axis=-1, keepdims=True)
    normalized = (x - mean) / np.sqrt(variance + 1e-5)
    return normalized * gamma + beta
#mean(axis=-1) calculate the mean over feature dimention assume (T,D) we get (T,1), assume (B,T,D), we get (B,T,1)

def batch_norm_over_time(x, gamma, beta):
    """Normalize each channel using statistics over the T dimension."""
    mean = x.mean(axis=0, keepdims=True)
    variance = x.var(axis=0, keepdims=True)
    normalized = (x - mean) / np.sqrt(variance + 1e-5)
    return normalized * gamma + beta
#mean(axis=0) calculate the mean over feature dimention assume (T,D) we get (1,D), assume (B,T,D),  we get (1,1,D)

class Linear:
    def __init__(self, input_dim, output_dim):
        self.weight = np.random.randn(input_dim, output_dim) / np.sqrt(input_dim)
        self.bias = np.zeros(output_dim)

    def __call__(self, x):
        return x @ self.weight + self.bias


class LayerNorm:
    def __init__(self, dimension):
        self.gamma = np.ones(dimension)
        self.beta = np.zeros(dimension)

    def __call__(self, x):
        return layer_norm(x, self.gamma, self.beta)


## Convolution Primitives

We use two intentionally direct implementations:

1. `conv2d_valid_stride` slides a 2D kernel over time and frequency.
2. `depthwise_conv1d_same` slides a separate 1D kernel over each feature channel.

The loops make the operations slower than optimized libraries, but they keep the tensor operations visible.


In [ ]:
def conv2d_valid_stride(x, weight, bias, stride=2):
    """
    x:      [T, F, input_channels]
    weight: [kernel_t, kernel_f, input_channels, output_channels]
    return: [new_T, new_F, output_channels]
    """
    kernel_t, kernel_f, _, output_channels = weight.shape
    output_t = (x.shape[0] - kernel_t) // stride + 1
    output_f = (x.shape[1] - kernel_f) // stride + 1
    output = np.zeros((output_t, output_f, output_channels))

    for t in range(output_t):
        for f in range(output_f):
            patch = x[
                t * stride : t * stride + kernel_t,
                f * stride : f * stride + kernel_f,
            ]
            output[t, f] = (
                np.tensordot(patch, weight, axes=([0, 1, 2], [0, 1, 2]))
                + bias
            )

    return output


def depthwise_conv1d_same(x, kernel):
    """
    x:      [T, D]
    kernel: [kernel_size, D]
    return: [T, D]
    """
    kernel_size = kernel.shape[0]
    padding = kernel_size // 2
    padded_x = np.pad(x, ((padding, padding), (0, 0)))
    output = np.zeros_like(x)

    for t in range(x.shape[0]):
        window = padded_x[t : t + kernel_size]
        output[t] = np.sum(window * kernel, axis=0)

    return output


## Step 1: Four-times Conv2D Subsampling

Each Conv2D layer has kernel size 3 and stride 2. With valid padding, one dimension changes according to

$$
L_{out} = \left\lfloor\frac{L_{in} - 3}{2}\right\rfloor + 1.
$$

After two layers, the time and frequency dimensions are each approximately one quarter of their original size. We flatten the remaining frequency and channel axes, then project to the model dimension `D`.


In [ ]:
class Conv2dSubsampling4:
    def __init__(self, input_dim, d_model, channels=4):
        self.conv1_weight = (
            np.random.randn(3, 3, 1, channels) / np.sqrt(3 * 3)
        )
        self.conv1_bias = np.zeros(channels)

        self.conv2_weight = (
            np.random.randn(3, 3, channels, channels)
            / np.sqrt(3 * 3 * channels)
        )
        self.conv2_bias = np.zeros(channels)

        frequency_after_conv1 = (input_dim - 3) // 2 + 1
        frequency_after_conv2 = (frequency_after_conv1 - 3) // 2 + 1
        flattened_dim = frequency_after_conv2 * channels
        self.projection = Linear(flattened_dim, d_model)

    def __call__(self, x):
        x = x[:, :, None]  # [T, F] -> [T, F, 1]
        x = conv2d_valid_stride(
            x, self.conv1_weight, self.conv1_bias, stride=2
        )
        x = np.maximum(x, 0)

        x = conv2d_valid_stride(
            x, self.conv2_weight, self.conv2_bias, stride=2
        )
        x = np.maximum(x, 0)

        time, frequency, channels = x.shape
        x = x.reshape(time, frequency * channels)
        return self.projection(x)


sample_features = np.random.randn(80, 80)
sample_subsampler = Conv2dSubsampling4(input_dim=80, d_model=16)
sample_encoded = sample_subsampler(sample_features)

print("Before subsampling:", sample_features.shape)
print("After subsampling: ", sample_encoded.shape)


## Step 2: Macaron Feed-forward Module

A Conformer block contains **two** feed-forward modules: one before attention and one near the end. Each is wrapped in a residual connection with weight `0.5`:

$$
x \leftarrow x + \tfrac{1}{2}\,\mathrm{FFN}(x).
$$

This “half-step” arrangement is often called a Macaron structure.


In [ ]:
class FeedForwardModule:
    def __init__(self, d_model, hidden_dim):
        self.norm = LayerNorm(d_model)
        self.linear1 = Linear(d_model, hidden_dim)
        self.linear2 = Linear(hidden_dim, d_model)

    def __call__(self, x):
        x = self.norm(x)
        x = self.linear1(x)
        x = swish(x)
        return self.linear2(x)


## Step 3: Relative Multi-head Self-attention

The model splits `D` features into `H` heads. Each head builds a `[T, T]` attention matrix, so every frame can gather information from every other frame.

The usual content score is

$$
\mathrm{score}(i,j) = \frac{q_i k_j^\top}{\sqrt{d_{head}}}.
$$

For clarity, this notebook adds a learned bias based on clipped relative distance `j - i`. This demonstrates relative position information, but it is simpler than the full relative-position formulation used in the original Conformer.


In [ ]:
class RelativeMultiHeadSelfAttention:
    def __init__(self, d_model, num_heads, max_relative_distance=16):
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.max_relative_distance = max_relative_distance

        self.norm = LayerNorm(d_model)
        self.query_projection = Linear(d_model, d_model)
        self.key_projection = Linear(d_model, d_model)
        self.value_projection = Linear(d_model, d_model)
        self.output_projection = Linear(d_model, d_model)

        self.relative_bias = (
            np.random.randn(num_heads, 2 * max_relative_distance + 1) * 0.01
        )
        self.last_attention = None

    def __call__(self, x):
        x = self.norm(x)
        time = x.shape[0]

        query = self.query_projection(x)
        key = self.key_projection(x)
        value = self.value_projection(x)

        # [T, D] -> [H, T, head_dim]
        query = query.reshape(time, self.num_heads, self.head_dim).transpose(1, 0, 2)
        key = key.reshape(time, self.num_heads, self.head_dim).transpose(1, 0, 2)
        value = value.reshape(time, self.num_heads, self.head_dim).transpose(1, 0, 2)

        # One [T, T] score matrix per head.
        scores = (query @ key.transpose(0, 2, 1)) / np.sqrt(self.head_dim)

        positions = np.arange(time)
        relative_positions = positions[None, :] - positions[:, None]
        relative_positions = np.clip(
            relative_positions,
            -self.max_relative_distance,
            self.max_relative_distance,
        )
        relative_positions += self.max_relative_distance
        scores = scores + self.relative_bias[:, relative_positions]

        attention = softmax(scores, axis=-1)
        self.last_attention = attention
        output = attention @ value

        # [H, T, head_dim] -> [T, D]
        output = output.transpose(1, 0, 2).reshape(time, -1)
        return self.output_projection(output)


## Step 4: Conformer Convolution Module

Attention is strong at long-range relationships, while this module emphasizes local time patterns:

```text
LayerNorm → pointwise projection → GLU
          → depthwise time convolution
          → batch normalization → Swish
          → pointwise projection
```

The gated linear unit (GLU) splits `[T, 2D]` into `a` and `b`, then computes `a * sigmoid(b)`. The depthwise convolution treats every channel separately, which keeps the operation easy to inspect.


In [ ]:
class ConformerConvModule:
    def __init__(self, d_model, kernel_size=7):
        self.norm = LayerNorm(d_model)
        self.pointwise1 = Linear(d_model, 2 * d_model)
        self.depthwise_kernel = (
            np.random.randn(kernel_size, d_model) / np.sqrt(kernel_size)
        )
        self.batch_norm_gamma = np.ones(d_model)
        self.batch_norm_beta = np.zeros(d_model)
        self.pointwise2 = Linear(d_model, d_model)

    def __call__(self, x):
        x = self.norm(x)
        x = self.pointwise1(x)  # [T, D] -> [T, 2D]

        a, b = np.split(x, 2, axis=-1)
        x = a * sigmoid(b)

        x = depthwise_conv1d_same(x, self.depthwise_kernel)
        x = batch_norm_over_time(
            x, self.batch_norm_gamma, self.batch_norm_beta
        )
        x = swish(x)
        return self.pointwise2(x)


## Step 5: Assemble One Conformer Block

All four modules preserve `[T, D]`, so residual additions are shape-compatible:

```text
x + 0.5 × FFN
  → x + self-attention
  → x + convolution
  → x + 0.5 × FFN
  → final LayerNorm
```


In [ ]:
class ConformerBlock:
    def __init__(
        self,
        d_model,
        hidden_dim,
        num_heads,
        convolution_kernel=7,
    ):
        self.ffn1 = FeedForwardModule(d_model, hidden_dim)
        self.attention = RelativeMultiHeadSelfAttention(d_model, num_heads)
        self.convolution = ConformerConvModule(
            d_model, convolution_kernel
        )
        self.ffn2 = FeedForwardModule(d_model, hidden_dim)
        self.final_norm = LayerNorm(d_model)

    def __call__(self, x):
        x = x + 0.5 * self.ffn1(x)
        x = x + self.attention(x)
        x = x + self.convolution(x)
        x = x + 0.5 * self.ffn2(x)
        return self.final_norm(x)


## Step 6: Complete Conformer-CTC Model

The final linear layer maps each encoded frame from `D` features to one score per vocabulary item. These scores are **CTC logits**. A trained model would normally use them in a CTC loss during training.


In [ ]:
class ConformerCTC:
    def __init__(
        self,
        input_dim=80,
        d_model=16,
        hidden_dim=32,
        num_heads=4,
        num_blocks=2,
        vocabulary_size=12,
    ):
        self.subsampling = Conv2dSubsampling4(input_dim, d_model)
        self.encoder = [
            ConformerBlock(
                d_model=d_model,
                hidden_dim=hidden_dim,
                num_heads=num_heads,
                convolution_kernel=7,
            )
            for _ in range(num_blocks)
        ]
        self.ctc_head = Linear(d_model, vocabulary_size)

    def __call__(self, features, verbose=True):
        if verbose:
            print("Input features:    ", features.shape)

        x = self.subsampling(features)
        if verbose:
            print("After subsampling: ", x.shape)

        for index, block in enumerate(self.encoder, start=1):
            x = block(x)
            if verbose:
                print(f"After block {index}:   ", x.shape)

        logits = self.ctc_head(x)
        if verbose:
            print("CTC logits:        ", logits.shape)

        return logits


## Step 7: Greedy CTC Decoding

Greedy decoding has three ideas:

1. Pick the highest-logit token at each frame.
2. Merge tokens repeated in adjacent frames.
3. Remove the blank token.

Order matters. For example, `[A, blank, A]` becomes `AA`, because the blank separates the two copies.


In [ ]:
def ctc_greedy_decode(logits, blank_id=0):
    frame_tokens = np.argmax(logits, axis=-1)

    result = []
    previous_token = None
    for token in frame_tokens:
        if token != blank_id and token != previous_token:
            result.append(int(token))
        previous_token = token

    return result


# A hand-made path makes the collapse rule easy to see.
# Token 0 is blank.
example_path = np.array([1, 1, 0, 1, 2, 2, 0])
example_logits = np.full((len(example_path), 3), -10.0)
example_logits[np.arange(len(example_path)), example_path] = 10.0

print("Frame-level path:", example_path.tolist())
print("CTC result:      ", ctc_greedy_decode(example_logits, blank_id=0))


## Run the Complete Model

Speech utterances naturally contain different numbers of frames. We process three lengths one at a time and observe how the attention matrices and CTC outputs adapt to each subsampled length.

The decoded IDs below are arbitrary because the model has not been trained.


In [ ]:
model = ConformerCTC(
    input_dim=80,
    d_model=16,
    hidden_dim=32,
    num_heads=4,
    num_blocks=2,
    vocabulary_size=12,
)

utterances = [
    np.random.randn(80, 80),
    np.random.randn(125, 80),
    np.random.randn(200, 80),
]

all_logits = []
for index, features in enumerate(utterances, start=1):
    print(f"\n===== Utterance {index} =====")
    logits = model(features)
    all_logits.append(logits)
    print("Decoded token IDs: ", ctc_greedy_decode(logits))


## Lightweight Checks

These checks focus on the two most important invariants:

- Softmax probabilities sum to one for every output frame.
- Each attention row sums to one over the frames it can attend to.


In [ ]:
last_logits = all_logits[-1]
ctc_probabilities = softmax(last_logits, axis=-1)
attention = model.encoder[-1].attention.last_attention

print("Last CTC output shape:       ", last_logits.shape)
print("First CTC probability sum:  ", ctc_probabilities[0].sum())
print("Last attention shape:        ", attention.shape)
print("First attention row sum:     ", attention[0, 0].sum())


## Key Takeaways

### What this implementation shows

- **Subsampling saves work:** two stride-2 convolutions shorten the time sequence by about four times.
- **Attention supplies global context:** every encoded frame can interact with every other encoded frame.
- **Convolution supplies local structure:** depthwise time convolution focuses on nearby speech patterns.
- **Macaron feed-forward layers refine each frame:** two half-weighted residual FFNs surround the central modules.
- **CTC handles unequal lengths:** frame-level predictions collapse into a shorter token sequence without a supplied frame-to-token alignment.

### Deliberate simplifications

- The model has random weights and implements only inference.
- There is no backpropagation, optimizer, training loop, or CTC loss.
- Relative position is represented by a learned distance bias, not the paper's full formulation.
- Batch normalization uses statistics from the current utterance.
- Inputs are processed one utterance at a time, so padding and masks are not needed.
- The direct NumPy convolutions favor readability over speed.

These simplifications keep the notebook focused on the architecture and tensor flow. A production system would use an optimized deep-learning framework, batched masking, learned parameters, and a real CTC training objective.
